<a href="https://colab.research.google.com/github/Ronglawan/AI-in-Financial-Reporting-and-Fraud-Detection/blob/main/AI_in_Financial_Reporting_and_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. โหลดข้อมูลดิบทั้งหมด
file_name = "Synthetic_Financial_datasets_log.csv"
df = pd.read_csv(file_name)
print(f"--- โหลดข้อมูลดิบเรียบร้อย ---")
print(f"จำนวนข้อมูลเริ่มต้น: {df.shape[0]:,} แถว | {df.shape[1]} คอลัมน์")

# ==================== 🛠️ จุดที่แก้ไขเพิ่มเติม 🛠️ ====================
# ลบเฉพาะแถวที่คอลัมน์เป้าหมาย 'isFraud' ดันเป็นค่าว่างออกก่อน (ป้องกัน Error Input y contains NaN)
df = df.dropna(subset=['isFraud'])
# ==================================================================

# 2. แยก Features (X) และ Label เป้าหมาย (y)
X = df.drop(columns=['isFraud', 'isFlaggedFraud'], errors='ignore')
y = df['isFraud']

# 3. ลบคอลัมน์ตามสเปกผู้สร้าง (ป้องกันข้อมูลหลอก และป้องกันเครื่องค้าง)
columns_to_drop = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'nameOrig', 'nameDest']
X = X.drop(columns=columns_to_drop, errors='ignore')

# 4. แปลงประเภทธุรกรรม (คอลัมน์ type) ให้เป็นตัวเลขด้วย One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# 5. ทำ Data Splitting แบ่งข้อมูลเป็น Train 80% และ Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. ทำ Feature Scaling (ปรับมาตรวัดของค่า step และ amount ให้สมดุลกัน)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- สรุปสถานะข้อมูลหลังแก้ไขรอบนี้ ---")
print(f"Train set (ข้อมูลสมบูรณ์) = {X_train_scaled.shape[0]:,} แถว")
print(f"Test set (ข้อมูลสมบูรณ์)  = {X_test_scaled.shape[0]:,} แถว")

# Build & Train Deep Learning

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. ออกแบบโครงสร้างสถาปัตยกรรมโมเดลสำหรับข้อมูลระดับล้านแถว (ANN Model)
def build_fraud_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(), # ช่วยจัดสรรโครงสร้างข้อมูลขนาดใหญ่ให้เสถียร
        layers.Dropout(0.3),         # ป้องกันโมเดลจดจำแพตเทิร์นเก่าจน Overfit
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid') # ทายผลลัพธ์สุดท้ายเป็น 0 หรือ 1
    ])
    return model

# 2. Compile โมเดล กำหนดตัววัดผล
model = build_fraud_model(input_shape=(X_train_scaled.shape[1],))
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 3. เริ่มรันกระบวนการ Train โมเดลด้วยชุดข้อมูลจริง 2 ล้านแถว
print("--- เริ่มต้นการฝึกสอนโมเดล Deep Learning (Training) ---")
history = model.fit(X_train_scaled, y_train,
                    epochs=10,
                    batch_size=1024, # ปรับความกว้างการป้อนข้อมูลให้ประมวลผลเร็วขึ้น
                    validation_data=(X_test_scaled, y_test))

# Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. ทำนายผลพยากรณ์จากข้อมูลทดสอบ (Test Set)
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

# 2. พิมพ์รายงานสถิติวัดผลเชิงวิชาการ
print("\n--- รายงานประเมินผล (Classification Report) ---")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

# 3. พล็อตตาราง Confusion Matrix ออกมาเป็นภาพกราฟสวยๆ
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
plt.ylabel('Actual Status')
plt.xlabel('Predicted Status')
plt.title('Confusion Matrix - Fraud Detection Model')
plt.show()

# ดูโครงสร้างข้อมูล

In [ ]:
# ดูประเภทข้อมูลและจำนวนช่องว่าง (Missing Values) ในแต่ละคอลัมน์
print("--- โครงสร้างของข้อมูล (Data Info) ---")
df.info()

print("\n" + "="*50 + "\n")

# ดูสถิติเชิงพรรณนา (ค่าเฉลี่ย, มิน-แมกซ์, ตัวเลขสรุปภาพรวม) ของคอลัมน์ตัวเลขทั้งหมด
print("--- สถิติภาพรวมของข้อมูล (Data Description) ---")
df.describe()

In [ ]:
# ส่องดูข้อมูล 10 แถวแรกสุดของไฟล์
#df.head(10)

# ส่องดูข้อมูล 10 แถวท้ายสุดของไฟล์
df.tail(2507040)

# สุ่ม (Random) แถวตรงไหนก็ได้ในไฟล์มาดู 10 แถวเพื่อเช็กความหลากหลาย
#df.sample(10)

In [ ]:
# ดึงเฉพาะธุรกรรมที่เป็นเคสทุจริต (isFraud เป็น 1) มาโชว์ดู 5 แถวแรก
fraud_cases = df[df['isFraud'] == 1]
print(f"จำนวนเคสโกงทั้งหมดที่ตรวจพบในไฟล์: {fraud_cases.shape[0]:,} แถว")
fraud_cases.head(5)

In [ ]:
# ต้องชัวร์ว่ามีไลบรารีอิมพอร์ตครบก่อนรัน
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# 1. เตรียมโมเดลที่จะใช้เปรียบเทียบเทคนิคจัดการ Imbalance
# แบบที่ 1: ใช้ค่าน้ำหนัก balanced (แบบที่เราเพิ่งทำไป)
model_balanced = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=50, max_depth=10, n_jobs=-1)
model_balanced.fit(X_train, y_train)
y_proba_balanced = model_balanced.predict_proba(X_test)[:, 1]

# แบบที่ 2: ใช้เทคนิคสร้างข้อมูลจำลอง SMOTE เข้ามาช่วย
print("กำลังทำ Oversampling ด้วย SMOTE...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

model_smote = RandomForestClassifier(random_state=42, n_estimators=50, max_depth=10, n_jobs=-1)
model_smote.fit(X_train_smote, y_train_smote)
y_proba_smote = model_smote.predict_proba(X_test)[:, 1]

# 2. คำนวณค่าและพล็อตกราฟ Precision-Recall Curve
plt.figure(figsize=(10, 7))

# พลอตเส้นของวิธีคุมน้ำหนัก (Cost-Sensitive)
precision_b, recall_b, _ = precision_recall_curve(y_test, y_proba_balanced)
auc_b = auc(recall_b, precision_b)
plt.plot(recall_b, precision_b, label=f'Random Forest (Cost-Sensitive, AUPRC = {auc_b:.4f})')

# พลอตเส้นของวิธี SMOTE
precision_s, recall_s, _ = precision_recall_curve(y_test, y_proba_smote)
auc_s = auc(recall_s, precision_s)
plt.plot(recall_s, precision_s, label=f'Random Forest (SMOTE, AUPRC = {auc_s:.4f})')

# ตกแต่งกราฟ PR Curve ให้ได้มาตรฐาน
plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision (Positive Predictive Value)')
plt.title('Precision-Recall (PR) Curve Comparison under Severe Imbalance')
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)

# เซฟรูปภาพกราฟขั้นสูงไปแปะใส่ตัวเล่ม
plt.savefig('pr_curve_comparison.png', dpi=300)
plt.show()

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

print("--- Step 1: Scaling Data for Anomaly Detection ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# คำนวณหาอัตราการปนเปื้อน (สัดส่วนเคสโกงจริงในข้อมูล เพื่อตั้งค่าโมเดล)
contamination_rate = sum(y_train) / len(y_train)

print(f"--- Step 2: Training Isolation Forest (Contamination: {contamination_rate:.4f}) ---")
iso_forest = IsolationForest(contamination=contamination_rate, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_scaled)

print("\n--- Step 3: Evaluation ---")
# Isolation Forest จะพ่นค่าออกมาเป็น: 1 = ปกติ, -1 = ผิดปกติ (Anomaly)
y_pred_iso = iso_forest.predict(X_test_scaled)

# แปลงค่า -1 ให้เป็นเลข 1 (Fraud) และเลข 1 ให้เป็นเลข 0 (Normal) เพื่อเทียบสถิติ
y_pred_iso_cleaned = [1 if x == -1 else 0 for x in y_pred_iso]

print("\n[Isolation Forest - Classification Report]")
print(classification_report(y_test, y_pred_iso_cleaned, digits=4))

In [ ]:
import shap
import matplotlib.pyplot as plt

print("--- Step 1: Initializing SHAP Explainer ---")
# ใช้ shap.Explainer แบบมาตรฐาน ซึ่งทำงานร่วมกับ Random Forest ได้เสถียรกว่า
explainer = shap.Explainer(model, X_train)

# สุ่มดึงตัวอย่างธุรกรรมในชุดทดสอบมา 200 แถวเพื่อความเร็วในการคำนวณ
X_test_sample = X_test.sample(200, random_state=42)
shap_values = explainer(X_test_sample)

print("--- Step 2: Plotting SHAP Summary Plot ---")
plt.figure(figsize=(10, 6))

# ตรวจสอบโครงสร้างมิติของข้อมูลก่อนพล็อตเพื่อป้องกัน Error
# หากเป็น Binary Classification: shap_values มักจะมีมิติเป็น (samples, features, classes)
if len(shap_values.shape) == 3:
    # เลือกเฉพาะ Class 1 (Fraud) ในมิติที่ 3 [:, :, 1]
    shap.plots.beeswarm(shap_values[:, :, 1], max_display=10, show=False)
else:
    # หากเวอร์ชันของไลบรารียุบรวมมิติมาให้แล้ว สามารถพล็อตตรงๆ ได้เลย
    shap.plots.beeswarm(shap_values, max_display=10, show=False)

plt.title("SHAP Bee-Swarm Plot: Inside the Mind of Fraud Detection AI", fontsize=14, pad=20)
plt.tight_layout()

# เซฟรูปภาพกราฟขั้นสูงไปแปะใส่ตัวเล่ม
plt.savefig('shap_fraud_explanation.png', dpi=300)
plt.show()